In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler

In [ ]:
df = pd.read_csv("university_cleaned.csv")

print(df.shape)
df.head()

(22125, 38)


,university_id,year,university_name,world_rank,national_rank,overall_score,country,region,city,university_type,...,degree_level,undergraduate_count,postgraduate_count,country_avg_rank,universities_ranked_count,best_university_rank,country_avg_overall_score,country_avg_academic_reputation,country_avg_citations,country_avg_international_ratio
0,1534,2018,Pontificia Universidad Católica Argentina,364.0,4.0,33.0000,Argentina,Latin America,Buenos Aires,Private,...,Undergraduate & Postgraduate,9560,5660,652.65,17,75.0,33.63,33.37,34.59,83.01
1,1269,2023,National Chi Nan University,1642.0,42.0,15.7275,Taiwan,Asia,Taiwan,Private,...,Undergraduate & Postgraduate,3452,2321,1041.45,47,77.0,26.27,20.02,19.55,8.87
2,299,2026,Canterbury Christ Church University,1300.5,113.0,29.4325,United Kingdom,Europe,Bexleyheath,Public,...,Undergraduate & Postgraduate,15183,8212,614.48,126,2.0,46.87,35.82,38.92,35.29
3,600,2023,Gazi University,1226.0,28.0,23.7450,Turkey,Asia,Turkey,Private,...,Undergraduate & Postgraduate,28683,12675,1264.96,73,460.0,24.69,17.31,16.31,8.88
4,2878,2016,University of Oulu,399.0,6.0,35.4300,Finland,Europe,Oulu,Public,...,Undergraduate & Postgraduate,9480,4576,352.89,9,76.0,38.96,38.59,40.11,9.00


In [ ]:
scaler = MinMaxScaler()

columns_to_normalize = [
    'overall_score',
    'academic_reputation_score',
    'employer_reputation_score',
    'citations_score',
    'research_output_score',
    'h_index',
    'faculty_to_student_ratio',
    'international_student_ratio'
]

normalized_columns = [col + '_norm' for col in columns_to_normalize]

df[normalized_columns] = scaler.fit_transform(df[columns_to_normalize]) * 100

In [ ]:
df['academic_excellence_index'] = (
    0.4 * df['overall_score_norm']
    + 0.3 * df['academic_reputation_score_norm']
    + 0.3 * df['employer_reputation_score_norm']
)

In [ ]:
df['research_impact_index'] = (
    0.5 * df['citations_score_norm']
    + 0.3 * df['research_output_score_norm']
    + 0.2 * df['h_index_norm']
)

In [ ]:
df['student_experience_index'] = (
    0.5 * df['faculty_to_student_ratio_norm']
    + 0.5 * df['international_student_ratio_norm']
)

In [ ]:
df['university_competitiveness_score'] = (
    0.35 * df['academic_excellence_index']
    + 0.35 * df['research_impact_index']
    + 0.30 * df['student_experience_index']
)

In [ ]:
score_scaler = MinMaxScaler()

df['university_competitiveness_score'] = (
    score_scaler.fit_transform(
        df[['university_competitiveness_score']]
    ) * 100
)

In [ ]:
max_rank = df['world_rank'].max()
min_rank = df['world_rank'].min()

df['global_ranking_score'] = (
    (max_rank - df['world_rank'])
    / (max_rank - min_rank)
) * 100

In [ ]:
df['performance_category'] = pd.qcut(
    df['university_competitiveness_score'],
    q=5,
    labels=[
        'Emerging',
        'Average',
        'Good',
        'Excellent',
        'Elite'
    ]
)

In [ ]:
kpis = [
    'academic_excellence_index',
    'research_impact_index',
    'student_experience_index',
    'university_competitiveness_score',
    'global_ranking_score'
]

print(df[kpis].describe().round(2))

print("\nPerformance Categories:\n")
print(df['performance_category'].value_counts())

       academic_excellence_index  research_impact_index  \
count                   22125.00               22125.00   
mean                       32.37                  26.91   
std                        17.25                  17.05   
min                         0.00                   0.02   
25%                        19.81                  14.93   
50%                        28.66                  24.28   
75%                        40.28                  35.87   
max                       100.00                  89.45   

       student_experience_index  university_competitiveness_score  \
count                  22125.00                          22125.00   
mean                       9.85                             34.48   
std                        3.91                             16.75   
min                        0.01                              0.00   
25%                        7.26                             22.05   
50%                        9.65                       

In [ ]:
df.to_csv("university_final_with_kpis.csv", index=False)

print("✅ Final dataset exported successfully.")

✅ Final dataset exported successfully.


In [ ]:
norm_cols = [col for col in df.columns if col.endswith('_norm')]
df.drop(columns=norm_cols, inplace=True)

In [ ]:
df.to_csv("university_final_with_kpis.csv", index=False)